# Stage3_experiment:normaliztion

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# # Adjust this path if your repo is stored elsewhere in Drive.
# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [2]:
# # Install Python dependencies (run once per session)
# !pip install -r {PROJECT_ROOT}/requirements.txt -q
# !python -m spacy download en

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/COMP5329_Assignment1-main
!ls
import os
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/COMP5329_Assignment1-main
assignment1.ipynb  Losses	       requirements.txt
_data		   _model	       Schedulers
Data		   _model_dropout_off  STAGE12_CODE_CHANGES_BY_MODULE.md
EvaluateTools	   _model_dropout_on   STAGE1_DEBUG_LOG.md
_log		   _model_group_norm   STAGE2_DEBUG_LOG.md
_log_dropout_off   _model_layer_norm   Tools
_log_dropout_on    Models	       TrainTools
_log_group_norm    Optimizers
_log_layer_norm    README.md
/content/drive/MyDrive/COMP5329_Assignment1-main


In [4]:
#  Install Python dependencies (run once per session)
!pip install -r requirements.txt
!python -m spacy download en

⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 150.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [5]:
import sys, os
# local root
from pathlib import Path
PROJECT_ROOT = str(Path.cwd().resolve())

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: /content/drive/MyDrive/COMP5329_Assignment1-main


---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [8]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 1000,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "none",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 200/200 [00:13<00:00, 14.79it/s]


STEP      200  loss 1828.407932



100%|██████████| 150/150 [00:02<00:00, 63.63it/s]


VALID(train) loss 34.328647  F1 6.931743  EM 0.000000



100%|██████████| 150/150 [00:02<00:00, 64.00it/s]


...

STEP     1000  loss 209.969667



100%|██████████| 150/150 [00:02<00:00, 64.18it/s]


VALID(train) loss 16.593790  F1 6.368547  EM 0.166667



100%|██████████| 150/150 [00:02<00:00, 64.38it/s]


TEST        loss 16.977461  F1 5.552651  EM 0.166667

Learning rate: [0.001]
Training finished.  Best F1: 6.1436  Best EM: 0.1667
Best F1: 6.1436  |  Best EM: 0.1667


Experiment: LayerNorm(A) vs GroupNorm(B)

Experiment A

In [9]:
from TrainTools.train import train

results_layer_norm = train(
    # data paths
    train_npz        = "_data/train.npz",
    dev_npz          = "_data/dev.npz",
    word_emb_json    = "_data/word_emb.json",
    char_emb_json    = "_data/char_emb.json",
    train_eval_json  = "_data/train_eval.json",
    dev_eval_json    = "_data/dev_eval.json",
    save_dir         = "_model_layer_norm",
    log_dir          = "_log_layer_norm",
    ckpt_name        = "model.pt",

    # training loop
    batch_size       = 8,
    num_steps        = 1000,
    checkpoint       = 200,
    val_num_batches  = 150,
    test_num_batches = 150,
    seed             = 42,

    # optimization
    optimizer_name   = "sgd",
    scheduler_name   = "none",
    loss_name        = "qa_nll",

    # normalization experiment
    norm_name        = "layer_norm",
    norm_groups      = 8,
)

print(f"LayerNorm | Best F1: {results_layer_norm['best_f1']:.4f} | Best EM: {results_layer_norm['best_em']:.4f}")

100%|██████████| 200/200 [00:13<00:00, 15.05it/s]


STEP      200  loss 1828.407932



100%|██████████| 150/150 [00:02<00:00, 63.97it/s]


VALID(train) loss 34.328647  F1 6.931743  EM 0.000000



100%|██████████| 150/150 [00:02<00:00, 64.31it/s]


...

STEP     1000  loss 209.969667



100%|██████████| 150/150 [00:02<00:00, 64.58it/s]


VALID(train) loss 16.593790  F1 6.368547  EM 0.166667



100%|██████████| 150/150 [00:02<00:00, 64.70it/s]


TEST        loss 16.977461  F1 5.552651  EM 0.166667

Learning rate: [0.001]
Training finished.  Best F1: 6.1436  Best EM: 0.1667
LayerNorm | Best F1: 6.1436 | Best EM: 0.1667


Experiment B

In [10]:
from TrainTools.train import train

results_group_norm = train(
    # data paths
    train_npz        = "_data/train.npz",
    dev_npz          = "_data/dev.npz",
    word_emb_json    = "_data/word_emb.json",
    char_emb_json    = "_data/char_emb.json",
    train_eval_json  = "_data/train_eval.json",
    dev_eval_json    = "_data/dev_eval.json",
    save_dir         = "_model_group_norm",
    log_dir          = "_log_group_norm",
    ckpt_name        = "model.pt",

    # training loop
    batch_size       = 8,
    num_steps        = 1000,
    checkpoint       = 200,
    val_num_batches  = 150,
    test_num_batches = 150,
    seed             = 42,

    # optimization
    optimizer_name   = "sgd",
    scheduler_name   = "none",
    loss_name        = "qa_nll",

    # normalization experiment
    norm_name        = "group_norm",
    norm_groups      = 8,
)

print(f"GroupNorm | Best F1: {results_group_norm['best_f1']:.4f} | Best EM: {results_group_norm['best_em']:.4f}")

100%|██████████| 200/200 [00:13<00:00, 14.53it/s]


STEP      200  loss 2315.971373



100%|██████████| 150/150 [00:03<00:00, 49.34it/s]


VALID(train) loss 40.194893  F1 6.763148  EM 0.000000



100%|██████████| 150/150 [00:03<00:00, 49.40it/s]


...

STEP     1000  loss 285.378939



100%|██████████| 150/150 [00:03<00:00, 49.34it/s]


VALID(train) loss 19.774089  F1 6.081912  EM 0.083333



100%|██████████| 150/150 [00:03<00:00, 49.43it/s]


TEST        loss 20.463413  F1 5.293317  EM 0.000000

Learning rate: [0.001]
Training finished.  Best F1: 6.0921  Best EM: 0.0833
GroupNorm | Best F1: 6.0921 | Best EM: 0.0833


In [11]:
print("===== Training Summary =====")
print(f"LayerNorm | Best F1: {results_layer_norm['best_f1']:.4f} | Best EM: {results_layer_norm['best_em']:.4f}")
print(f"GroupNorm | Best F1: {results_group_norm['best_f1']:.4f} | Best EM: {results_group_norm['best_em']:.4f}")

===== Training Summary =====
LayerNorm | Best F1: 6.1436 | Best EM: 0.1667
GroupNorm | Best F1: 6.0921 | Best EM: 0.0833


---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [12]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|██████████| 1309/1309 [00:20<00:00, 64.68it/s]


TEST  loss 17.574258  F1 7.551860  EM 0.200669
F1: 7.5519  |  EM: 0.2007  |  Loss: 17.574258


Custom evaluation function

During evaluation, we encountered a parameter mismatch issue when loading checkpoints trained with different normalization strategies. This occurred because the original evaluation function did not support configurable normalization methods and defaulted to LayerNorm.

To resolve this, we implemented a custom evaluation function that explicitly passes the normalization configuration (e.g., GroupNorm) when reconstructing the model. This ensures consistency between training and evaluation, allowing the model parameters to be loaded correctly.

In [13]:
import os
import argparse
import torch
import ujson as json

from Data import SQuADDataset, load_dev_eval, load_word_char_mats
from Losses import losses
from Models import QANet
from EvaluateTools.eval_utils import run_eval

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def evaluate_norm(
    dev_npz="_data/dev.npz",
    word_emb_json="_data/word_emb.json",
    char_emb_json="_data/char_emb.json",
    dev_eval_json="_data/dev_eval.json",
    save_dir="_model",
    log_dir="_log",
    ckpt_name="model.pt",
    batch_size=8,
    test_num_batches=-1,
    loss_name="qa_nll",
    para_limit=400,
    ques_limit=50,
    char_limit=16,
    d_model=96,
    num_heads=8,
    glove_dim=300,
    char_dim=64,
    dropout=0.1,
    dropout_char=0.05,
    pretrained_char=False,
    norm_name="layer_norm",
    norm_groups=8,
    activation="relu",
    init_name="kaiming",
):
    os.makedirs(log_dir, exist_ok=True)

    args = argparse.Namespace(
        dev_npz=dev_npz,
        word_emb_json=word_emb_json,
        char_emb_json=char_emb_json,
        dev_eval_json=dev_eval_json,
        para_limit=para_limit,
        ques_limit=ques_limit,
        char_limit=char_limit,
        d_model=d_model,
        num_heads=num_heads,
        glove_dim=glove_dim,
        char_dim=char_dim,
        dropout=dropout,
        dropout_char=dropout_char,
        pretrained_char=pretrained_char,
        norm_name=norm_name,
        norm_groups=norm_groups,
        activation=activation,
        init_name=init_name,
    )

    word_mat, char_mat = load_word_char_mats(args)
    model = QANet(word_mat, char_mat, args).to(DEVICE)

    dev_eval = load_dev_eval(args)
    dev_dataset = SQuADDataset(dev_npz)

    ckpt_path = os.path.join(save_dir, ckpt_name)
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])

    metrics, ans = run_eval(
        model,
        dev_dataset,
        dev_eval,
        num_batches=test_num_batches,
        batch_size=batch_size,
        use_random_batches=False,
        device=DEVICE,
        loss_fn=losses[loss_name],
    )

    with open(os.path.join(log_dir, "answers.json"), "w") as f:
        json.dump(ans, f)

    print("TEST  loss {loss:.6f}  F1 {f1:.6f}  EM {exact_match:.6f}".format(**metrics))
    return {
        "f1": metrics["f1"],
        "exact_match": metrics["exact_match"],
        "loss": metrics["loss"],
    }

Evaluate A

In [14]:
metrics_layer_norm = evaluate_norm(
    dev_npz        = "_data/dev.npz",
    word_emb_json  = "_data/word_emb.json",
    char_emb_json  = "_data/char_emb.json",
    dev_eval_json  = "_data/dev_eval.json",
    save_dir       = "_model_layer_norm",
    log_dir        = "_log_layer_norm",
    ckpt_name      = "model.pt",
    norm_name      = "layer_norm",
    norm_groups    = 8,
)

print(f"LayerNorm | F1: {metrics_layer_norm['f1']:.4f} | EM: {metrics_layer_norm['exact_match']:.4f} | Loss: {metrics_layer_norm['loss']:.4f}")

100%|██████████| 1309/1309 [00:20<00:00, 64.61it/s]


TEST  loss 17.574258  F1 7.551860  EM 0.200669
LayerNorm | F1: 7.5519 | EM: 0.2007 | Loss: 17.5743


Evaluate B

In [15]:
metrics_group_norm = evaluate_norm(
    dev_npz        = "_data/dev.npz",
    word_emb_json  = "_data/word_emb.json",
    char_emb_json  = "_data/char_emb.json",
    dev_eval_json  = "_data/dev_eval.json",
    save_dir       = "_model_group_norm",
    log_dir        = "_log_group_norm",
    ckpt_name      = "model.pt",
    norm_name      = "group_norm",
    norm_groups    = 8,
)

print(f"GroupNorm | F1: {metrics_group_norm['f1']:.4f} | EM: {metrics_group_norm['exact_match']:.4f} | Loss: {metrics_group_norm['loss']:.4f}")

100%|██████████| 1309/1309 [00:26<00:00, 49.40it/s]


TEST  loss 21.382730  F1 7.596441  EM 0.057334
GroupNorm | F1: 7.5964 | EM: 0.0573 | Loss: 21.3827


In [16]:
print("===== Final Comparison =====")
print(f"LayerNorm | Train Best F1: {results_layer_norm['best_f1']:.4f} | Train Best EM: {results_layer_norm['best_em']:.4f} | Eval F1: {metrics_layer_norm['f1']:.4f} | Eval EM: {metrics_layer_norm['exact_match']:.4f}")
print(f"GroupNorm | Train Best F1: {results_group_norm['best_f1']:.4f} | Train Best EM: {results_group_norm['best_em']:.4f} | Eval F1: {metrics_group_norm['f1']:.4f} | Eval EM: {metrics_group_norm['exact_match']:.4f}")

===== Final Comparison =====
LayerNorm | Train Best F1: 6.1436 | Train Best EM: 0.1667 | Eval F1: 7.5519 | Eval EM: 0.2007
GroupNorm | Train Best F1: 6.0921 | Train Best EM: 0.0833 | Eval F1: 7.5964 | Eval EM: 0.0573


# Experiment: Effect of Normalization Strategy on QANet

## 1. Research Question  

How does the choice of normalization method (LayerNorm vs GroupNorm) affect model performance and generalization in the repaired QANet?

---

## 2. Hypothesis  

We hypothesize that **LayerNorm** will outperform **GroupNorm**, as it is more suitable for sequence-based architectures and is widely used in transformer-style models.

---

## 3. Experimental Setup  

We conduct a controlled experiment by modifying only the normalization strategy while keeping all other hyperparameters fixed.

- Model: repaired QANet  
- Dataset: SQuAD v1.1  
- Batch size: 8  
- Training steps: 1000  
- Optimizer: SGD  
- Scheduler: none  
- Loss: QA NLL  
- Seed: 42  

Two configurations were evaluated:

- **LayerNorm**
- **GroupNorm (8 groups)**

To ensure consistency between training and evaluation, a **custom evaluation function** was implemented so that the normalization configuration used during evaluation matches the training setup.

---

## 4. Results  

| Setting | Train F1 | Train EM | Eval F1 | Eval EM |
|--------|--------|--------|--------|--------|
| LayerNorm | 6.1436 | 0.1667 | 7.5519 | **0.2007** |
| GroupNorm | 6.0921 | 0.0833 | **7.5964** | 0.0573 |

---

## 5. Analysis  

The results reveal a clear trade-off between LayerNorm and GroupNorm.

LayerNorm achieves a significantly higher Exact Match (EM) score, indicating that it produces more fully correct answers. In contrast, GroupNorm achieves a slightly higher F1 score, suggesting that its predictions are closer to the ground truth on average, but not necessarily exact.

This difference highlights that the two normalization strategies influence model behavior in different ways. LayerNorm appears to produce more precise predictions, while GroupNorm tends to generate answers that are partially correct but less accurate overall.

The large gap in EM suggests that LayerNorm is more reliable when exact answer boundaries are required, which is particularly important in question answering tasks.

Although GroupNorm slightly outperforms LayerNorm in F1, the improvement is marginal compared to the substantial drop in EM. Therefore, from a practical perspective, LayerNorm demonstrates better overall performance.

---

## 6. Conclusion  

This experiment shows that normalization strategy significantly affects model behavior. While GroupNorm achieves a slightly higher F1 score, LayerNorm produces substantially higher Exact Match scores, indicating better precision in predicting exact answer spans. Therefore, **LayerNorm is a more suitable normalization method for the repaired QANet**.

---

## 7. Limitations and Future Work  

This experiment is conducted with a fixed set of hyperparameters and a single random seed, which may limit the generality of the conclusions. Additionally, the training budget is relatively small, and the model performance is still far from optimal.

Future work could explore:

- Multiple random seeds to improve robustness  
- Longer training steps for more stable convergence  
- Other normalization techniques (e.g., BatchNorm)  
- Interaction with other components such as dropout and optimizer choice  

---

## 8. Additional Insight  

This experiment highlights that **architectural design choices, such as normalization strategy, can significantly influence model behavior and performance**, sometimes more than simple hyperparameter tuning.